# Week 11 — Contextual Bandits and Off-Policy Evaluation · Workbook

This notebook contains the 9 graded exercises summarised in `notes.pdf`.
Each exercise has three cells:
1. **Prompt** (markdown) with a difficulty tag (★ / ★★ / ★★★).
2. **Hint** (collapsible markdown) — try yourself first.
3. **Scaffold** (code) — fill in your code, then run the **test** cell.

Reference solutions live in `project/solutions.py`. The test cells assert
your output matches the reference; when running this notebook end-to-end
under papermill we substitute scaffolds with reference calls.


**Teaching companion:** [Worked examples, derivations, analogies, and self-checks](teaching_guide.md). Read the relevant example before attempting its exercise.

In [ ]:
# Standard preamble.
import sys, os
sys.path.insert(0, os.path.abspath('..'))   # for common/
sys.path.insert(0, os.path.abspath('.'))    # for project/
from common.plotting import use_rl_style, RL_PALETTE
from common.seed import seed_everything
use_rl_style(); seed_everything(0)
import numpy as np
from project import solutions as sol
from project.env import SyntheticLeadBandit
from project.agents import (
    UCB1, KLUCB, BetaThompson, LinUCB, LinTS, EpsilonGreedyLogger, kl_bernoulli,
)
from project.ope import ips, snips, doubly_robust, switch_dr, RidgeRewardModel, true_policy_value
from project.train import softmax_policy, uniform_policy, greedy_oracle_policy


## Exercise 1 — UCB1 bonus (★)

Compute the UCB1 exploration bonus  $\sqrt{2 \log t / n_a}$  for $t = 100$ and $n_a = 10$.

> **Hint** (try yourself first): one numpy call: `np.sqrt(2*np.log(t)/n_a)`.

In [ ]:
# Reference solution (filled in for end-to-end execution)
ans1 = sol.exercise_1_ucb_bonus(t=100, n_a=10)
assert abs(ans1 - np.sqrt(2*np.log(100)/10)) < 1e-9
print('E1 bonus =', ans1)


## Exercise 2 — KL-UCB upper bound (★)

Implement the KL-UCB upper-confidence solver: given $\hat\mu_a, n_a, t$, return
$\max\{q \in [\hat\mu_a, 1] : n_a \cdot \mathrm{KL}(\hat\mu_a, q) \le \log t + 3 \log\log t\}$
via bisection.

> **Hint**: the function $q \mapsto \mathrm{KL}(\hat\mu_a, q)$ is monotone increasing on $[\hat\mu_a, 1]$ — perfect for bisection.

In [ ]:
ub_a = sol.exercise_2_klucb_upper(0.4, n_a=20, t=200)
ub_b = sol.exercise_2_klucb_upper(0.4, n_a=200, t=200)
assert ub_a > 0.4 and ub_b > 0.4 and ub_b < ub_a
print('KL-UCB upper bound (n=20):', round(ub_a, 4))
print('KL-UCB upper bound (n=200):', round(ub_b, 4))


## Exercise 3 — Thompson Sampling regret (★★)

Run Bernoulli Thompson Sampling on a 3-armed bandit with means
$[0.2, 0.5, 0.8]$ for $T = 1000$ rounds. Report cumulative regret.
The Lai–Robbins lower bound is roughly
$(\Delta_1/\mathrm{KL}(0.2,0.8) + \Delta_2/\mathrm{KL}(0.5,0.8)) \log T$;
TS should be in the right order of magnitude.

> **Hint**: use `BetaThompson(K=3, seed=0)` from `project.agents`.

In [ ]:
reg = sol.exercise_3_thompson_regret(T=1000, seed=0)
print('TS cumulative regret:', round(reg, 2))
assert reg < 0.6 * 1000


## Exercise 4 — LinUCB on synthetic insurance (★★)

Instantiate `SyntheticLeadBandit(seed=0)` and `LinUCB(K=6, d=8, alpha=1.0)`,
roll out for $T = 1000$ rounds, and report cumulative regret.

> **Hint**: see the LinUCB pseudo-code in `notes.pdf` chapter 4; the project module already implements it.

In [ ]:
reg = sol.exercise_4_linucb_regret(T=1000, seed=0)
print('LinUCB regret:', round(reg, 2))
assert reg < 0.5 * 1000


## Exercise 5 — IPS and SNIPS for a uniform target (★★)

Collect $n=500$ logged samples from a 0.3-greedy logger; compute the IPS
and SNIPS estimates of the uniform target policy.

> **Hint**: the uniform target is just `np.full(K, 1/K)`. SNIPS divides by the sum of weights.

In [ ]:
v_ips, v_snips = sol.exercise_5_ips_snips_uniform(n=500, seed=0)
print('IPS  =', round(v_ips, 3))
print('SNIPS =', round(v_snips, 3))
# Uniform target has true value = mean over arms of mean reward; should be in [0, 1]
assert 0.0 <= v_snips <= 1.0


## Exercise 6 — Doubly Robust against softmax target (★★)

Fit a ridge reward model on the same log, then evaluate the softmax-true
policy with DR. Compare to the true (on-policy) value.

> **Hint**: `RidgeRewardModel(K, d).fit(log)` returns a callable `qhat(x) -> np.ndarray of shape (K,)`.

In [ ]:
v_dr = sol.exercise_6_dr_softmax(n=1200, seed=0)
env = SyntheticLeadBandit(seed=0)
v_true = true_policy_value(env, softmax_policy(env, beta=4.0), n=4000, seed=42)
print('DR    =', round(v_dr, 3))
print('truth =', round(v_true, 3))
assert abs(v_dr - v_true) < 0.25


## Exercise 7 — Effective sample size (★)

Compute the effective sample size of a weight vector. Sanity-check:
$\mathrm{ESS}(\mathbf{1}) = n$, $\mathrm{ESS}([10, 0, \ldots, 0]) \approx 1$.

> **Hint**: $\mathrm{ESS} = (\sum_i w_i)^2 / \sum_i w_i^2$.

In [ ]:
w = np.ones(100)
assert abs(sol.exercise_7_ess(w) - 100.0) < 1e-9
w2 = np.array([10.0] + [0.0] * 99)
print('ESS uniform:', sol.exercise_7_ess(w))
print('ESS spiked:', sol.exercise_7_ess(w2))


## Exercise 8 — SWITCH-DR threshold sweep (★★)

Vary $\tau \in \{1, 5, 10, 50, 1000\}$ for the oracle-greedy target. Plot
estimate vs.\ $\tau$ and identify the bias-variance trade-off.

> **Hint**: small $\tau$ → mostly direct method (biased, low variance); large $\tau$ → recovers DR.

In [ ]:
taus = [1.0, 5.0, 10.0, 50.0, 1000.0]
estimates = [sol.exercise_8_switch_dr(n=800, tau=tau, seed=0) for tau in taus]
for tau, est in zip(taus, estimates):
    print(f'tau={tau:7.1f}  V_SWITCH-DR = {est:.3f}')
# At least some variation across tau
assert max(estimates) - min(estimates) >= 0.0


## Exercise 9 — synthetic insurance OPE memo (★★★, open-ended)

For three counterfactual policies (uniform, softmax-true, oracle-greedy) compute:
- The on-policy ground truth (we have it because the env is synthetic).
- SNIPS and DR estimates from a 1500-row 0.3-greedy log.

Identify which estimator is more reliable for each policy and write a short
recommendation.

> **Hint**: Inspect the observed weights, ESS, held-out reward-model errors, and uncertainty. A peaky policy can have poor overlap, but neither the sign of SNIPS error nor the best estimator is guaranteed.

In [ ]:
truth, snips_est, dr_est = sol.exercise_9_synthetic_ope(n=1500, seed=0)
print(f'softmax-true on-policy truth = {truth:.3f}')
print(f'SNIPS estimate              = {snips_est:.3f}  (err = {abs(snips_est-truth):.3f})')
print(f'DR    estimate              = {dr_est:.3f}  (err = {abs(dr_est-truth):.3f})')
# DR should be at least competitive with SNIPS (within 0.25 tolerance)
assert abs(dr_est - truth) <= abs(snips_est - truth) + 0.25


## Wrap-up

You have:
- Implemented UCB1, KL-UCB, Thompson, LinUCB.
- Built the synthetic insurance lead-broker bandit and trained LinUCB to sub-linear regret.
- Collected $\varepsilon$-greedy logs *with propensities*.
- Run IPS, SNIPS, DR, and SWITCH-DR off-policy evaluations.
- Diagnosed the importance-weight distribution via ESS.

Next week: ranking, two-sided matching, and fairness.

<!-- AUTO-HINTS-SECTION -->
## Hints toward answers

Attempt the exercise first. Before opening a hint, write the quantity you are
estimating, the data source, and the expected update direction. The
[teaching guide](teaching_guide.md) supplies worked examples, assumptions,
and transfer questions. The hints below collect the exercise's own inline
hints; they do not reveal or execute reference solutions.

<details>
<summary>Exercise 1 — UCB1 bonus (★)</summary>

> **Hint** (try yourself first): one numpy call: `np.sqrt(2*np.log(t)/n_a)`.

</details>

<details>
<summary>Exercise 2 — KL-UCB upper bound (★)</summary>

> **Hint**: the function $q \mapsto \mathrm{KL}(\hat\mu_a, q)$ is monotone increasing on $[\hat\mu_a, 1]$ — perfect for bisection.

</details>

<details>
<summary>Exercise 3 — Thompson Sampling regret (★★)</summary>

> **Hint**: use `BetaThompson(K=3, seed=0)` from `project.agents`.

</details>

<details>
<summary>Exercise 4 — LinUCB on synthetic insurance (★★)</summary>

> **Hint**: see the LinUCB pseudo-code in `notes.pdf` chapter 4; the project module already implements it.

</details>

<details>
<summary>Exercise 5 — IPS and SNIPS for a uniform target (★★)</summary>

> **Hint**: the uniform target is just `np.full(K, 1/K)`. SNIPS divides by the sum of weights.

</details>

<details>
<summary>Exercise 6 — Doubly Robust against softmax target (★★)</summary>

> **Hint**: `RidgeRewardModel(K, d).fit(log)` returns a callable `qhat(x) -> np.ndarray of shape (K,)`.

</details>

<details>
<summary>Exercise 7 — Effective sample size (★)</summary>

> **Hint**: $\mathrm{ESS} = (\sum_i w_i)^2 / \sum_i w_i^2$.

</details>

<details>
<summary>Exercise 8 — SWITCH-DR threshold sweep (★★)</summary>

> **Hint**: small $\tau$ → mostly direct method (biased, low variance); large $\tau$ → recovers DR.

</details>

<details>
<summary>Exercise 9 — synthetic insurance OPE memo (★★★, open-ended)</summary>

> **Hint**: Inspect the observed weights, ESS, held-out reward-model errors, and uncertainty. A peaky policy can have poor overlap, but neither the sign of SNIPS error nor the best estimator is guaranteed.

</details>

After solving, change one input or assumption and predict the result before
running it. If a cell calls `project.solutions`, passing it verifies the
reference demonstration; test your own implementation separately.
